In [ ]:
import pandas as pd


# 1. Load data (Ensure files are uploaded to the folder icon on the left!)
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
category_name = pd.read_csv('product_category_name_translation.csv')

# 2. Merge systematically using 'left' to keep all orders
df = pd.merge(orders, items, on='order_id', how='left')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, payments, on='order_id', how='left')
df = pd.merge(df, reviews, on='order_id', how='left')
df = pd.merge(df, category_name, on='product_category_name', how='left')

# 3. Check the result
print(f"Final dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
print(df.head())
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
product_category_name = pd.read_csv('product_category_name_translation.csv')

df = pd.merge(orders, items, on='order_id', how='left')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, order_payments, on='order_id', how='left')
df = pd.merge(df, order_reviews, on='order_id', how='left')
df = pd.merge(df, product_category_name, on='product_category_name', how='left')
print(df.head())

# Group by zip code to get the average lat/lon before merging
geo_unique = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

# Now it is safe to merge with your main 'df'
df = pd.merge(df, geo_unique, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

In [ ]:
# 1. Handle Missing Values
# Remove rows where delivery date is missing (likely canceled or pending)
df = df.dropna(subset=['order_delivered_customer_date'])

# Fill missing product category names with 'unknown'
df['product_category_name'] = df['product_category_name'].fillna('unknown')

# Fill missing freight_value with 0
df['freight_value'] = df['freight_value'].fillna(0)


# 2. Filter Data
# Keep only 'delivered' orders
df = df[df['order_status'] == 'delivered']

# Remove orders with negative prices or freight
df = df[(df['price'] >= 0) & (df['freight_value'] >= 0)]

# Filter for the specific date range (2017-01-01 to 2018-08-31)
# We convert the column to datetime first to make filtering possible
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df = df[(df['order_purchase_timestamp'] >= '2017-01-01') &
        (df['order_purchase_timestamp'] <= '2018-08-31')]

# Remove duplicate order_id entries to ensure unique transactions
df = df.drop_duplicates(subset=['order_id'])


# 3. Data Type Conversion
# Convert all date columns to datetime objects
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    df[col] = pd.to_datetime(df[col])

# Ensure IDs are strings and prices are floats
df['order_id'] = df['order_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)
df['product_id'] = df['product_id'].astype(str)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print("Data Cleaning Complete!")
print(f"Remaining rows: {len(df)}")

In [ ]:
# 1. Total number of orders
total_orders = df['order_id'].nunique()

# 2. Total revenue (sum of all prices)
total_revenue = df['price'].sum()

# 3. Average Order Value (AOV)
aov = total_revenue / total_orders

# 4. Average items per order
# We count total rows (items) divided by unique order IDs
avg_items_per_order = len(df) / total_orders

# 5. Date range of data
date_min = df['order_purchase_timestamp'].min()
date_max = df['order_purchase_timestamp'].max()

print(f"Total Orders: {total_orders}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"AOV: ${aov:.2f}")
print(f"Date Range: {date_min} to {date_max}")

In [ ]:
# Create a Year-Month column for grouping
df['month_year'] = df['order_purchase_timestamp'].dt.to_period('M')

# Grouping by month
monthly_trends = df.groupby('month_year').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

# Renaming columns for clarity
monthly_trends.columns = ['Month', 'Number_of_Orders', 'Total_Revenue']
monthly_trends['Average_Order_Value'] = monthly_trends['Total_Revenue'] / monthly_trends['Number_of_Orders']

# Calculate Month-over-Month Growth Rate
# Formula: ((Current - Previous) / Previous) * 100
monthly_trends['Growth_Rate'] = monthly_trends['Number_of_Orders'].pct_change() * 100

print(monthly_trends)

In [ ]:
# Extract Day of Week and Hour
df['day_of_week'] = df['order_purchase_timestamp'].dt.day_name()
df['hour'] = df['order_purchase_timestamp'].dt.hour

# Daily Pattern
daily_pattern = df.groupby('day_of_week')['order_id'].nunique().sort_values(ascending=False)
print("Orders by Day of Week:")
print(daily_pattern)

# Hourly Pattern
hourly_pattern = df.groupby('hour')['order_id'].nunique()
peak_hour = hourly_pattern.idxmax()

print(f"\nPeak Ordering Hour: {peak_hour}:00")

In [ ]:
# 1. Create the Category Summary Table
category_analysis = df.groupby('product_category_name_english').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

# 2. Calculate Required Metrics
# Rename columns for clarity
category_analysis.columns = ['Category', 'Number_of_Orders', 'Total_Revenue']

# Calculate Average Price per category
category_analysis['Average_Price'] = category_analysis['Total_Revenue'] / category_analysis['Number_of_Orders']

# Calculate Percentage of Total Orders
total_orders_count = category_analysis['Number_of_Orders'].sum()
category_analysis['Percentage_of_Total_Orders'] = (category_analysis['Number_of_Orders'] / total_orders_count) * 100

# 3. Identify Top 10 Categories by Volume
top_10_volume = category_analysis.sort_values(by='Number_of_Orders', ascending=False).head(10)

# 4. Identify Top 10 Categories by Revenue
top_10_revenue = category_analysis.sort_values(by='Total_Revenue', ascending=False).head(10)

# Display Results
print("--- Top 10 Categories by Volume ---")
print(top_10_volume[['Category', 'Number_of_Orders', 'Percentage_of_Total_Orders']])

print("\n--- Top 10 Categories by Revenue ---")
print(top_10_revenue[['Category', 'Total_Revenue', 'Average_Price']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting Top 10 by Volume
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10_volume, x='Number_of_Orders', y='Category', palette='viridis')
plt.title('Top 10 Product Categories by Order Volume')
plt.show()

In [ ]:
# Create State Summary
state_analysis = df.groupby('customer_state').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

# Rename and Calculate Metrics
state_analysis.columns = ['State', 'Number_of_Orders', 'Total_Revenue']
state_analysis['Percentage_of_Total_Orders'] = (state_analysis['Number_of_Orders'] / state_analysis['Number_of_Orders'].sum()) * 100
state_analysis['Average_Order_Value'] = state_analysis['Total_Revenue'] / state_analysis['Number_of_Orders']

# Identify Top and Bottom 5
top_5_states = state_analysis.sort_values(by='Number_of_Orders', ascending=False).head(5)
bottom_5_states = state_analysis.sort_values(by='Number_of_Orders', ascending=True).head(5)

print("--- Top 5 States by Volume ---")
print(top_5_states)

In [ ]:
# Seller State Summary
seller_analysis = df.groupby('seller_state').agg({
    'seller_id': 'nunique',
    'order_id': 'nunique'
}).reset_index()

seller_analysis.columns = ['Seller_State', 'Number_of_Sellers', 'Orders_Fulfilled']
seller_analysis['Avg_Orders_per_Seller'] = seller_analysis['Orders_Fulfilled'] / seller_analysis['Number_of_Sellers']

print("\n--- Seller Distribution ---")
print(seller_analysis.sort_values(by='Number_of_Sellers', ascending=False).head(5))

In [ ]:
# 1. Classify orders
df['delivery_type'] = df.apply(lambda x: 'in-state' if x['customer_state'] == x['seller_state'] else 'cross-state', axis=1)

# 2. Calculate Percentage of Cross-State Orders
cross_state_pct = (df['delivery_type'] == 'cross-state').mean() * 100

# 3. Calculate Delivery Time (in days)
# We subtract purchase date from delivered date
df['delivery_time_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# 4. Compare Delivery Times
delivery_comparison = df.groupby('delivery_type')['delivery_time_days'].mean()

print(f"\nPercentage of Cross-State Orders: {cross_state_pct:.2f}%")
print("\nAverage Delivery Time (Days):")
print(delivery_comparison)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extract the month (1-12) from the timestamp
df['month_num'] = df['order_purchase_timestamp'].dt.month

# 2. Calculate average order volume per month across all years
seasonality = df.groupby('month_num')['order_id'].nunique().reset_index()
seasonality.columns = ['Month', 'Total_Orders']

# 3. Create the Line Chart
plt.figure(figsize=(10, 5))
sns.lineplot(data=seasonality, x='Month', y='Total_Orders', marker='o', color='teal')
plt.title('Monthly Seasonality (Aggregated 2017-2018)')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Identify the peak
peak_month = seasonality.loc[seasonality['Total_Orders'].idxmax(), 'Month']
print(f"The seasonal peak occurs in Month: {peak_month}")

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# 1. Prepare data: Monthly order counts
trend_data = df.groupby('month_year')['order_id'].nunique().reset_index()
trend_data['Month_Index'] = np.arange(len(trend_data)) # Create numeric x-axis (0, 1, 2...)

# 2. Fit a Linear Trend Line
X = trend_data[['Month_Index']]
y = trend_data['order_id']
model = LinearRegression().fit(X, y)
trend_line = model.predict(X)

# 3. Plot the Actual vs. Trend
plt.figure(figsize=(12, 6))
plt.plot(trend_data['Month_Index'], y, label='Actual Orders', marker='s')
plt.plot(trend_data['Month_Index'], trend_line, label='Linear Trend', linestyle='--', color='red')
plt.title('Order Trend Analysis (2017-01 to 2018-08)')
plt.xlabel('Months since Jan 2017')
plt.ylabel('Number of Orders')
plt.legend()
plt.show()

# Calculate the slope to see if growing
slope = model.coef_[0]
print(f"Trend Slope: {slope:.2f} orders per month.")
if slope > 0:
    print("Conclusion: Orders are increasing over time.")
else:
    print("Conclusion: Orders are decreasing over time.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visual style for all plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Convert 'Month' column to string for plotting purposes
monthly_trends['Month_str'] = monthly_trends['Month'].astype(str)
seasonality['Month_str'] = seasonality['Month'].astype(str)

# 1. Monthly Order Trend (Line Chart)
plt.figure()
sns.lineplot(data=monthly_trends, x='Month_str', y='Number_of_Orders', marker='o', color='b')
plt.title('Monthly Order Trend (2017-2018)', fontsize=15)
plt.xlabel('Month') # Add x-label for clarity after conversion
plt.xticks(rotation=45)
plt.show()

# 2. Revenue Trend (Line Chart)
plt.figure()
sns.lineplot(data=monthly_trends, x='Month_str', y='Total_Revenue', marker='s', color='g')
plt.title('Monthly Revenue Trend', fontsize=15)
plt.xlabel('Month') # Add x-label for clarity after conversion
plt.xticks(rotation=45)
plt.show()

# 3. Day of Week Pattern (Bar Chart)
# Ensure days are in correct order
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.figure()
sns.barplot(x=daily_pattern.index, y=daily_pattern.values, order=day_order, palette='muted')
plt.title('Orders by Day of the Week', fontsize=15)
plt.ylabel('Number of Orders')
plt.show()

# 4. Top 10 Categories (Horizontal Bar Chart)
plt.figure()
sns.barplot(data=top_10_volume, x='Number_of_Orders', y='Category', palette='viridis')
plt.title('Top 10 Product Categories by Volume', fontsize=15)
plt.show()

# 5. Geographic Orders by State (Bar Chart / Heatmap)
plt.figure()
state_sorted = state_analysis.sort_values('Number_of_Orders', ascending=False)
sns.barplot(data=state_sorted, x='State', y='Number_of_Orders', palette='magma')
plt.title('Total Orders by State', fontsize=15)
plt.show()

# 6. Seasonality Chart (Month-wise Average)
plt.figure()
sns.lineplot(data=seasonality, x='Month_str', y='Total_Orders', marker='o', color='orange', linewidth=2.5)
plt.title('Seasonality: Average Orders per Month', fontsize=15)
plt.xlabel('Month') # Add x-label for clarity after conversion
plt.xticks(rotation=45) # Rotate for better visibility
plt.show()